In [4]:
import io

class BasicTokenizer:
    def __init__(self):
        self.merges= {}
        self.vocab = {}
        v = 0 
        while v <= 255:
            self.vocab[v] = bytes([v])
            v +=1

    def get_stats(self, tokens):
        mydict = {}
        for i in range(len(tokens) - 1) :
            currentTuple = (tokens[i], tokens[i + 1])
            if currentTuple in mydict :
                mydict[currentTuple] += 1
            else :
                mydict[currentTuple] = 1
        return mydict

    def merge(self, tokens, top_pair, new_id):
        new_tokens=[]
        t=0
        while t < len(tokens):
            if t < len(tokens) -1 and tokens[t] == top_pair[0] and tokens[t + 1] == top_pair[1]:
                    new_tokens.append(new_id)
                    t += 2
            else:
                new_tokens.append(tokens[t])
                t += 1
        return new_tokens

    def train(self, text, vocab_size):
        tokens = list(text.encode("utf-8"))
        num_merges = vocab_size - 256
        i = 0
        while i <= num_merges :
            mydict = self.get_stats(tokens)
            print(mydict)
            if not mydict: break
            top_pair = max(mydict, key=mydict.get)
            new_id = 256 + i 

            self.merges[top_pair] = new_id
            self.vocab[new_id] = self.vocab[top_pair[0]] + self.vocab[top_pair[1]]
            tokens = self.merge(tokens, top_pair, new_id)
            i += 1
        return tokens

    def encode(self, text):
        tokens = list(text.encode("utf-8"))
        while len(tokens) >=2:
            stats = self.get_stats(tokens)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            idx = self.merges[pair]
            tokens = self.merge(tokens, pair, idx)
        return tokens
        

    def decode(self, ids):
        b_parts = []
        for i in range(len(ids)) :
            b_parts.append(self.vocab[ids[i]])
    
        full_bytes = b"".join(b_parts)
        return full_bytes.decode("utf-8", errors="replace")

    def load(self):
        with open('example.txt', 'r', encoding='utf-8') as f:
            rawText = f.read()
        return rawText

if __name__ == "__main__":
    tokenizer = BasicTokenizer()
    rawText = tokenizer.load()
    tokenizer.train(rawText, 276)
    
    #encode & decode
    tokens = tokenizer.encode(rawText)
    print("Tokens compressés :", tokens[:10])
    
    texte_reconstruit = tokenizer.decode(tokens)
    print("Succès ?", rawText == texte_reconstruit)

{(65, 118): 1, (118, 97): 13, (97, 110): 66, (110, 116): 111, (116, 32): 112, (32, 100): 165, (100, 101): 86, (101, 32): 327, (32, 114): 15, (114, 101): 91, (101, 110): 189, (116, 114): 48, (101, 114): 96, (114, 32): 111, (100, 97): 15, (110, 115): 47, (115, 32): 148, (32, 108): 143, (108, 101): 117, (100, 117): 33, (117, 114): 46, (32, 101): 75, (101, 116): 39, (100, 226): 4, (226, 128): 30, (128, 153): 24, (153, 101): 12, (101, 120): 25, (120, 97): 1, (97, 109): 6, (109, 105): 17, (105, 110): 63, (110, 101): 30, (110, 32): 126, (100, 195): 50, (195, 169): 112, (169, 116): 10, (116, 97): 23, (97, 105): 60, (105, 108): 29, (108, 32): 26, (32, 109): 52, (109, 111): 31, (111, 100): 30, (195, 168): 27, (168, 108): 15, (32, 116): 74, (114, 97): 51, (115, 102): 8, (102, 111): 19, (111, 114): 39, (114, 109): 22, (109, 101): 54, (114, 44): 2, (44, 32): 17, (32, 112): 78, (112, 114): 34, (114, 195): 23, (169, 99): 20, (99, 195): 3, (169, 100): 3, (32, 97): 35, (97, 114): 36, (114, 116): 12, (1